# Modelos

Veamos la API de bajo nivel de Transformers: los modelos que envuelven el código de PyTorch de los propios transformers.

Este notebook puede ejecutarse en un runtime T4 gratuito o de bajo costo.

## Un recordatorio más

**Consejo:**

En medio de la ejecución de un Colab, puede aparecer un error como este:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

¡Es un mensaje de error súper engañoso! Por favor, no intentes cambiar las versiones de los paquetes...

Esto pasa porque Google cambió tu runtime de Colab, quizá porque Google Colab estaba muy ocupado. La solución es:

1. Menú Kernel >> Disconnect and delete runtime
2. Recarga el colab desde cero y ve al menú Edit >> Clear All Outputs
3. Conéctate a una nueva T4 con el botón de arriba a la derecha
4. Selecciona "View resources" en el menú de arriba a la derecha para confirmar que tienes GPU
5. Vuelve a ejecutar las celdas del colab, de arriba hacia abajo, empezando por las instalaciones de pip

Y todo debería funcionar muy bien. Si no, ¡pregúntame!

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

# Iniciar sesión en Hugging Face

1. Si todavía no lo has hecho, crea una cuenta GRATUITA de HuggingFace en https://huggingface.co y ve a Settings, luego Create a new API token, dándote permisos de escritura haciendo clic en la pestaña WRITE

2. Pulsa el ícono de "llave" en el panel lateral izquierdo y agrega un nuevo secreto:
`HF_TOKEN = tu_token`

3. Ejecuta la celda de abajo para iniciar sesión.

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

### Acceder a Llama

Ayer deberías haber recibido la aprobación para usar este modelo:

https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

Puedes usarlo hoy, o es más rápido si también te aprueban este otro modelo:

https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

Selecciona este enlace para ver si también necesitas solicitar aprobación. Elige la versión de Llama que quieras usar abajo comentando una de estas líneas. O sáltate Llama por completo.

In [ ]:
# Modelos instruct y 1 modelo de razonamiento

# Llama 3.1 es más grande y ya deberías estar aprobado
# mira aquí: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

# LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Llama 3.2 es más pequeño, pero quizá tengas que solicitar acceso de nuevo
# mira aquí: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [ ]:
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
  ]

# Acceder a Llama 3.1 de Meta

Para usar el fantástico Llama 3.1, Meta exige que aceptes sus términos de servicio.

Visita la página de instrucciones del modelo en Hugging Face:
https://huggingface.co/meta-llama/Meta-Llama-3.1-8B

En la parte superior de la página hay instrucciones para aceptar sus términos. Si es posible, usa el mismo correo que tu cuenta de Hugging Face.

En mi experiencia, la aprobación llega en un par de minutos. Una vez aprobado para cualquier modelo 3.1, aplica a toda la familia de modelos.

Si tienes problemas para acceder a Llama, consulta este colab, que incluye algunas sugerencias por si Meta no te aprueba por cualquier motivo.

https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8

In [ ]:
# Configuración de cuantización: esto permite cargar el modelo en memoria y usar menos memoria

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

Si la siguiente celda te da un error de permisos 403, revisa:
1. ¿Has iniciado sesión en HuggingFace? Prueba a ejecutar `login()` para comprobar que tu clave funciona
2. ¿Configuraste tu clave de API con permisos completos de lectura y escritura?
3. Si visitas la página de Llama 3.1 en https://huggingface.co/meta-llama/Meta-Llama-3.1-8B, ¿aparece cerca de la parte superior que tienes acceso al modelo?

Y sigue el colab de solución de problemas de Llama:

https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8

In [ ]:
# Tokenizador

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

In [ ]:
inputs

In [ ]:
# El modelo

model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

In [ ]:
memory = model.get_memory_footprint() / 1e6
print(f"Huella de memoria: {memory:,.1f} MB")

## Mirando por dentro el modelo Transformer

La siguiente celda imprime el objeto `model` de HuggingFace para Llama.

Este objeto de modelo es una red neuronal, implementada con el framework de Python PyTorch. La red usa la arquitectura inventada por científicos de Google en 2017: la arquitectura Transformer.

Aunque no vamos a profundizar en la teoría, es una oportunidad para tener algo de intuición sobre qué es realmente el Transformer.

Si eres totalmente nuevo en las redes neuronales, echa un vistazo a mi [lista de reproducción introductoria en YouTube](https://www.youtube.com/playlist?list=PLWHe-9GP9SMMdl6SLaovUQF2abiLGbMjs) para ver los fundamentos.

Ahora mira las capas de la red neuronal que se imprimen en la siguiente celda. Fíjate en esto:

- Consiste en capas
- Hay algo llamado "embedding": toma los tokens y los convierte en vectores de 4,096 dimensiones. Aprenderemos más sobre esto en la semana 5.
- Luego hay 16 grupos de capas (32 para Llama 3.1) llamados "Decoder layers". Cada Decoder layer contiene tres tipos de capa: (a) capas de self-attention, (b) capas perceptrón multicapa (MLP) y (c) capas de batch norm.
- Al final hay una capa LM Head, que produce la salida

Fíjate en que se menciona que el modelo está cuantizado a 4 bits.

No hace falta profundizar más en la teoría en este punto, pero si quieres hacerlo, le pedí a nuestro amigo en común que tomara esta impresión y creara un tutorial que recorre cada capa. También mira las dimensiones en cada punto. Si te interesa, trabaja con este tutorial después de ejecutar la siguiente celda:

https://chatgpt.com/canvas/shared/680cbea6de688191a20f350a2293c76b

In [ ]:
# Ejecuta esta celda, mira lo que se imprime e investiga las capas

model

### Y si quieres ir aún más profundo en los Transformers

Además de mirar cada una de las capas del modelo, puedes ver el código de HuggingFace que implementa Llama con PyTorch.

Aquí está el repositorio de HuggingFace Transformers:  
https://github.com/huggingface/transformers

Y dentro de él, el código de Llama 4:  
https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py

No es necesario entrar en este nivel de detalle: el trabajo de un AI Engineer es seleccionar, optimizar, hacer fine-tuning y aplicar LLM, no programar un Transformer en PyTorch. OpenAI, Meta y otros laboratorios de vanguardia gastaron millones en construir y entrenar estos modelos. ¡Pero es una madriguera fascinante si te interesa!

In [ ]:
# Bien, con eso, ¡ahora ejecutemos el modelo!

outputs = model.generate(inputs, max_new_tokens=80)
outputs[0]

In [ ]:
# ¡Eso no tiene mucho sentido!
# Mejor probemos esto..

tokenizer.decode(outputs[0])

In [ ]:
# Limpiar la memoria
# Gracias a Kuan L. por ayudarme a conseguir que libere la memoria correctamente
# Si seleccionas "Show Resources" arriba a la derecha para ver la memoria de la GPU, puede que no baje de inmediato
# Pero parece que la memoria sí queda disponible para los modelos nuevos que se cargan más adelante.

del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

## Un par de notas rápidas sobre el siguiente bloque de código:

Uso una utilidad de HuggingFace llamada TextStreamer para que los resultados se vayan mostrando en streaming.
Para hacer streaming de los resultados, simplemente reemplazamos:  
`outputs = model.generate(inputs, max_new_tokens=80)`  
por:  
`streamer = TextStreamer(tokenizer)`  
`outputs = model.generate(inputs, max_new_tokens=80, streamer=streamer)`

También añadí el argumento `add_generation_prompt=True` a mi llamada para crear la plantilla de chat. Esto asegura que Phi genere una respuesta a la pregunta, en lugar de solo predecir cómo continúa el prompt del usuario. Prueba a poner este valor en False para ver qué pasa. Puedes leer sobre este argumento aquí:

https://huggingface.co/docs/transformers/main/en/chat_templating#what-are-generation-prompts

¡Gracias al estudiante Piotr B. por plantear el problema!

In [ ]:
# Envolviendo todo en una función, con streaming y prompts de generación

def generate(model, messages, quant=True, max_new_tokens=80):
  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
  streamer = TextStreamer(tokenizer)
  if quant:
    model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)



In [ ]:
generate(PHI, messages)

## Acceder a Gemma de Google

Un estudiante me avisó (¡gracias, Alex K!) de que Google ahora también exige aceptar sus términos en HuggingFace antes de usar Gemma.

Visita la página del modelo en este enlace, confirma que aceptas sus términos y así se te concederá el acceso.

https://huggingface.co/google/gemma-3-270m-it

In [ ]:
messages = [
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
generate(GEMMA, messages, quant=False)

In [ ]:
generate(QWEN, messages)

In [ ]:
generate(DEEPSEEK, messages, quant=False, max_new_tokens=500)